In [16]:
# CODE FOR CELL 1 (REVISI KUOTA GLOBAL ASIMETRIS)
from curl_cffi import requests as requests_cffi
import time
import pandas as pd

daftar_keyword = [
    # --- KECANTIKAN & PERAWATAN DIRI ---
    "serum wajah", "sunscreen", "pelembap wajah", "sabun cuci muka", 
    "toner wajah", "micellar water", "masker wajah", "body lotion", 
    "lipstik matte", "lip tint", "bedak tabur", "foundation wajah",
    "parfum wanita", "parfum pria", "deodorant", "sampo anti ketombe", 
    "lulur mandi", "kapas kecantikan",

    # --- ELEKTRONIK & GADGET MURAH (Medan Perang Ulasan) ---
    "headset murah", "earphone murah", "kabel data murah", "charger murah",
    "powerbank murah", "jam tangan murah", "kipas angin mini", "tripod hp",
    "mouse wireless", "keyboard mekanik", "flashdisk 64gb", "micro sd murah" "stop kontak",

    # --- MAKANAN & MINUMAN ---
    "kopi bubuk", "kopi instan", "teh celup", "susu UHT", 
    "mie instan goreng", "mie instan kuah", "bumbu nasi goreng", 
    "kecap manis", "saus sambal", "minyak goreng", "camilan gurih", "sosis sapi", "nugget ayam",

    # --- PAKAIAN, FASHION & AKSESORIS MURAH ---
    "kaos polos murah", "dompet murah", "baju kemeja murah", "sandal slop murah", 
    "sepatu murah", "kaos kaki katun", "celana jeans pria", "celana kulot wanita", 
    "jaket hoodie", "tas ransel pria", "tas selempang wanita", "kacamata hitam",

    # --- KEBUTUHAN RUMAH TANGGA & ATK ---
    "deterjen cair", "pewangi pakaian", "sabun cuci piring", "handuk mandi", 
    "sprei kasur", "botol minum tumbler", "kotak makan plastik", "rak sepatu", 
    "buku tulis", "pulpen hitam", "pensil mekanik", "spidol papan tulis", "sticky notes",
    "rak jepit", "barang unik", "payung lipat"
]

# Target ~8750 per bintang agar total 4 kelas terkumpul 35.000 ulasan berimbang
TARGET_GLOBAL_PER_RATING = 8750 
TARGET_RATINGS = [1, 2, 3, 5]

kuota_global = {1: 0, 2: 0, 3: 0, 5: 0}
semua_sku = []
hasil_ulasan = []

headers_global = {
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.blibli.com/",
    "Connection": "keep-alive"
}

print(f"Sistem Siap! Menargetkan {TARGET_GLOBAL_PER_RATING} data per kelas untuk Bintang {TARGET_RATINGS}.")

Sistem Siap! Menargetkan 8750 data per kelas untuk Bintang [1, 2, 3, 5].


In [18]:
# CODE FOR CELL 2 (CARI & SIMPAN SKU KE FILE)
from curl_cffi import requests as requests_cffi
import time
import pandas as pd

semua_sku = []

print("=== MEMULAI TAHAP 1: PENCARIAN SKU (PRODUK TERLARIS) ===")

for keyword in daftar_keyword:
    print(f"Mencari produk terlaris untuk: '{keyword}'...")
    url_search = "https://www.blibli.com/backend/search/products"
    
    params_search = {
        "searchTerm": keyword,
        "sort": "16",
        "page": "1",
        "start": "0",
        "merchantSearch": "true",
        "multiCategory": "true",
        "intent": "true",
        "channelId": "mobile-web",
        "showFacet": "false",
        "userIdentifier": "1780992671.b384657e-8752-49ca-9255-3dbb70fe21c5", 
        "isMobileBCA": "false",
        "isJual": "false",
        "firstLoad": "false"
    }
    
    headers_aman = {
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
        "Referer": "https://www.blibli.com/",
        "Connection": "keep-alive"
    }
    
    try:
        res_search = requests_cffi.get(url_search, headers=headers_aman, params=params_search, impersonate="chrome110")
        if res_search.status_code != 200:
            continue
            
        data_search = res_search.json()
        jumlah_sku_baru = 0
        
        for item in data_search.get('data', {}).get('products', []):
            sku = item.get('sku')
            relative_url = item.get('url')
            
            if sku:
                full_url = f"https://www.blibli.com{relative_url}" if relative_url else f"https://www.blibli.com/p/p/ps--{sku}"
                semua_sku.append({
                    "keyword": keyword, 
                    "sku": sku,
                    "url_produk": full_url
                })
                jumlah_sku_baru += 1
                
    except Exception as e:
        print(f"  [!] Error pada keyword '{keyword}': {e}")
        
    time.sleep(1.5)

# --- PROSES DROP DUPLICATE & SIMPAN KE FILE ---
if semua_sku:
    df_sku = pd.DataFrame(semua_sku)
    
    total_awal = len(df_sku)
    # Drop duplicate berdasarkan kolom 'sku' agar benar-benar unik
    df_sku_clean = df_sku.drop_duplicates(subset=['sku'])
    total_akhir = len(df_sku_clean)
    
    print("\n=== TAHAP 1 SELESAI ===")
    print(f"Total SKU mentah yang ditemukan : {total_awal}")
    print(f"Total SKU unik setelah dibersihkan: {total_akhir} (Berhasil membuang {total_awal - total_akhir} duplikat)")
    
    # Simpan ke file CSV khusus tempat penyimpanan SKU
    df_sku_clean.to_csv("daftar_sku_terlaris.csv", index=False, encoding='utf-8')
    print("[V] Berhasil menyimpan file 'daftar_sku_terlaris.csv'")
else:
    print("[!] Tidak ada SKU yang berhasil didapatkan.")

=== MEMULAI TAHAP 1: PENCARIAN SKU (PRODUK TERLARIS) ===
Mencari produk terlaris untuk: 'serum wajah'...
Mencari produk terlaris untuk: 'sunscreen'...
Mencari produk terlaris untuk: 'pelembap wajah'...
Mencari produk terlaris untuk: 'sabun cuci muka'...
Mencari produk terlaris untuk: 'toner wajah'...
Mencari produk terlaris untuk: 'micellar water'...
Mencari produk terlaris untuk: 'masker wajah'...
Mencari produk terlaris untuk: 'body lotion'...
Mencari produk terlaris untuk: 'lipstik matte'...
Mencari produk terlaris untuk: 'lip tint'...
Mencari produk terlaris untuk: 'bedak tabur'...
Mencari produk terlaris untuk: 'foundation wajah'...
Mencari produk terlaris untuk: 'parfum wanita'...
Mencari produk terlaris untuk: 'parfum pria'...
Mencari produk terlaris untuk: 'deodorant'...
Mencari produk terlaris untuk: 'sampo anti ketombe'...
Mencari produk terlaris untuk: 'lulur mandi'...
Mencari produk terlaris untuk: 'kapas kecantikan'...
Mencari produk terlaris untuk: 'headset murah'...
Menc

In [19]:
# CODE FOR CELL 3 (BACA SKU DARI FILE & SCRAPE REVIEWS)
from curl_cffi import requests as requests_cffi
import time
import pandas as pd

# 1. Load data SKU unik dari file CSV hasil Tahap 1
try:
    df_sku_input = pd.read_csv("daftar_sku_terlaris.csv")
    semua_sku = df_sku_input.to_dict(orient="records")
    print(f"[V] Berhasil me-load {len(semua_sku)} SKU unik dari file 'daftar_sku_terlaris.csv'. Siap memproses ulasan...")
except FileNotFoundError:
    print("[!] File 'daftar_sku_terlaris.csv' tidak ditemukan! Silakan jalankan Cell 2 terlebih dahulu.")
    semua_sku = []

# 2. Jalankan Mesin Scraper (Hanya berjalan jika file SKU berhasil di-load)
if semua_sku:
    hasil_ulasan = []
    kuota_global = {1: 0, 2: 0, 3: 0, 5: 0}
    
    print("\n=== MEMULAI TAHAP 2: ASIMETRIS GLOBAL BUCKET SCRAPER ===")
    
    for item in semua_sku:
        keyword = item["keyword"]
        sku = item["sku"]
        url_produk = item["url_produk"]
        
        if all(kuota_global[b] >= TARGET_GLOBAL_PER_RATING for b in TARGET_RATINGS):
            print("\n[V] SELESAI! TARGET GLOBAL 35.000 DATA BERIMBANG SUDAH TERPENUHI!")
            break
            
        print(f"\n[+] Menyisir SKU: {sku} (Keyword asal: '{keyword}')")
        print(f"    Progres Kuota -> Bintang 1: {kuota_global[1]}/{TARGET_GLOBAL_PER_RATING} | Bintang 2: {kuota_global[2]}/{TARGET_GLOBAL_PER_RATING} | Bintang 3: {kuota_global[3]}/{TARGET_GLOBAL_PER_RATING} | Bintang 5: {kuota_global[5]}/{TARGET_GLOBAL_PER_RATING}")
        
        for bintang in TARGET_RATINGS:
            if kuota_global[bintang] >= TARGET_GLOBAL_PER_RATING:
                continue
                
            if bintang in [1, 2, 3]:
                max_ulasan_per_produk = 500  
            else:
                max_ulasan_per_produk = 20   
                
            halaman = 1
            ulasan_didapat = 0
            
            while ulasan_didapat < max_ulasan_per_produk and kuota_global[bintang] < TARGET_GLOBAL_PER_RATING:
                url_review = "https://www.blibli.com/backend/product-review/public-reviews"
                params_review = {
                    "itemPerPage": 30,
                    "page": halaman,
                    "productSku": sku,
                    "sortBy": "POPULAR",
                    "rating": bintang
                }
                
                headers_review = {
                    "Accept": "application/json, text/plain, */*",
                    "Referer": f"https://www.blibli.com/p/p/ps--{sku}"
                }
                
                try:
                    res_review = requests_cffi.get(url_review, headers=headers_review, params=params_review, impersonate="chrome110")
                    if res_review.status_code != 200:
                        break
                        
                    data_review = res_review.json()
                    
                    reviews = []
                    if isinstance(data_review, list):
                        reviews = data_review
                    elif isinstance(data_review, dict):
                        data_content = data_review.get('data')
                        if isinstance(data_content, list):
                            reviews = data_content
                        elif isinstance(data_content, dict):
                            reviews = data_content.get('reviews', [])
                    
                    if not reviews:
                        break
                        
                    for rev in reviews:
                        if kuota_global[bintang] < TARGET_GLOBAL_PER_RATING and ulasan_didapat < max_ulasan_per_produk:
                            teks_clean = str(rev.get('content', '')).replace('\n', ' ').strip()
                            
                            if len(teks_clean) > 0:
                                hasil_ulasan.append({
                                    "Keyword": keyword,
                                    "Product SKU": sku,
                                    "Product URL": url_produk,
                                    "Rating": rev.get('rating'),
                                    "Review": teks_clean
                                })
                                ulasan_didapat += 1
                                kuota_global[bintang] += 1
                                
                except Exception:
                    break
                    
                halaman += 1
                time.sleep(0.6)
                
            if ulasan_didapat > 0:
                print(f"    -> Memanen {ulasan_didapat} ulasan Bintang {bintang}")

    print("\n=== PROSES SCRAPING MASSAL SELESAI ===")

[V] Berhasil me-load 2788 SKU unik dari file 'daftar_sku_terlaris.csv'. Siap memproses ulasan...

=== MEMULAI TAHAP 2: ASIMETRIS GLOBAL BUCKET SCRAPER ===

[+] Menyisir SKU: BEF-44369-00436 (Keyword asal: 'serum wajah')
    Progres Kuota -> Bintang 1: 0/8750 | Bintang 2: 0/8750 | Bintang 3: 0/8750 | Bintang 5: 0/8750
    -> Memanen 1 ulasan Bintang 3
    -> Memanen 20 ulasan Bintang 5

[+] Menyisir SKU: PTP-60093-00130 (Keyword asal: 'serum wajah')
    Progres Kuota -> Bintang 1: 0/8750 | Bintang 2: 0/8750 | Bintang 3: 1/8750 | Bintang 5: 20/8750
    -> Memanen 4 ulasan Bintang 3
    -> Memanen 20 ulasan Bintang 5

[+] Menyisir SKU: SEB-60031-00432 (Keyword asal: 'serum wajah')
    Progres Kuota -> Bintang 1: 0/8750 | Bintang 2: 0/8750 | Bintang 3: 5/8750 | Bintang 5: 40/8750
    -> Memanen 3 ulasan Bintang 3
    -> Memanen 20 ulasan Bintang 5

[+] Menyisir SKU: ERS-60058-00036 (Keyword asal: 'serum wajah')
    Progres Kuota -> Bintang 1: 0/8750 | Bintang 2: 0/8750 | Bintang 3: 8/8750 

In [20]:
# CODE FOR CELL 4
if hasil_ulasan:
    # Mengubah list menjadi DataFrame Pandas
    df_final = pd.DataFrame(hasil_ulasan)
    
    print("=== HASIL AKHIR DISTRIBUSI DATASET PRIMER ===")
    print(f"Total baris data terkumpul: {len(df_final)}")
    
    # Cek jumlah baris per Rating bintang (Harus seimbang/rata)
    distribusi = df_final.groupby(['Rating']).size().reset_index(name='Total Data')
    display(distribusi)
    
    # Menyimpan file dataset
    nama_file_dataset = "dataset_blibli_berimbang_35k.csv"
    df_final.to_csv(nama_file_dataset, index=False, encoding='utf-8')
    print(f"\n[V] Dataset sukses diekspor ke: '{nama_file_dataset}'")
    
    # Mengintip contoh data teratas
    print("\nSamples Data:")
    display(df_final.head())
else:
    print("[!] Variabel hasil_ulasan kosong. Periksa kembali Tahap 1 atau koneksi internetmu.")

=== HASIL AKHIR DISTRIBUSI DATASET PRIMER ===
Total baris data terkumpul: 15420


,Rating,Total Data
0,1,2856
1,2,1001
2,3,2813
3,5,8750



[V] Dataset sukses diekspor ke: 'dataset_blibli_berimbang_35k.csv'

Samples Data:


,Keyword,Product SKU,Product URL,Rating,Review
0,serum wajah,BEF-44369-00436,https://www.blibli.com/p/nivea-sun-face-oil-co...,3,"Mantab cepet bgt nyampenya, langsung kurir bli..."
1,serum wajah,BEF-44369-00436,https://www.blibli.com/p/nivea-sun-face-oil-co...,5,Waktu pengemasan dan pengiriman baik. Barang j...
2,serum wajah,BEF-44369-00436,https://www.blibli.com/p/nivea-sun-face-oil-co...,5,"Pengiriman cepat, produknya ok, cepat meresap ..."
3,serum wajah,BEF-44369-00436,https://www.blibli.com/p/nivea-sun-face-oil-co...,5,"Pengiriman cepat, kirain udah packaging model ..."
4,serum wajah,BEF-44369-00436,https://www.blibli.com/p/nivea-sun-face-oil-co...,5,Ok. sip. Packaging aman. Pengiriman cepat. Pes...
